# ABANICCO — Interactive Color Analysis

**AB ANgular Illustrative Classification of COlor** ([paper](https://www.mdpi.com/1424-8220/23/6/3338))

This notebook mirrors the planned FIJI plugin workflow:

1. **Input** — choose an image and the number of shades per category
2. **General visualization** — category segmentations + interactive polar description
   (hover a bubble → radius, angle, L value and mixing name; hover the fan → % of that category in the image)
3. **Post-processing** — change boundaries, create clusters, change #colors, apply morphology
4. **Export** — polar description, segmentations, shade tables, and reproducible code

Requirements: `pip install numpy scikit-image scikit-learn matplotlib plotly ipywidgets pillow pandas`


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # so `abanicco` package is importable

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, clear_output

from abanicco import (analyze, load_regions, show_segmentations,
                      polar_description, shade_charts, interactive_polar,
                      make_cluster, custom_region, shades_to_records)

%matplotlib inline

## 1 · Input
Pick an image (or type a path) and the analysis parameters.

In [ ]:
IMAGE_DIR = '../../example_images' if os.path.isdir('../../example_images') else '.'
files = sorted(f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.tif', '.tiff')))

image_picker = W.Dropdown(options=files, description='Image')
path_box     = W.Text(description='...or path', placeholder='/full/path/to/image')
numshades    = W.IntSlider(value=15, min=2, max=30, description='# shades/cat.')
morpho       = W.IntSlider(value=10, min=0, max=200, description='Morphology')
run_btn      = W.Button(description='Analyze', button_style='primary')
out          = W.Output()

STATE = {}

def load_image():
    p = path_box.value.strip() or os.path.join(IMAGE_DIR, image_picker.value)
    return np.array(Image.open(p).convert('RGB')), p

def run(_=None):
    with out:
        clear_output()
        im, p = load_image()
        print(f'Analyzing {os.path.basename(p)}  ({im.shape[1]}x{im.shape[0]}) ...')
        regions, rows = analyze(im, numshades=numshades.value, morpho=morpho.value)
        STATE.update(im=im, regions=regions, rows=rows, path=p,
                     log=[f'analyze(numshades={numshades.value}, morpho={morpho.value})'])
        print(f'Done: {len(rows)} shades across {sum(1 for r in regions if r.shades)} categories.')
        print('Run the cells below to visualize.')

run_btn.on_click(run)
display(W.VBox([W.HBox([image_picker, path_box]), numshades, morpho, run_btn, out]))

# Auto-run once with the default selection so 'Run All' works end-to-end.
# Use the button above to re-run with different settings.
if 'im' not in STATE:
    run(None)


## 2 · General visualization
### 2.1 Category segmentations

In [ ]:
fig = show_segmentations(STATE['im'], STATE['regions'])
plt.show()

### 2.2 Interactive polar description
Two views of the same data: a **bubble plot** (marker size = % of the image) and a
**scatter plot** (fixed-size dots, clearer for dense shade clusters). Hover a marker
for its coordinates (radius, angle, L), % of image, and mixing name
(*x% of color 1 + y% of color 2*). Hover the outer **fan** for the % of each category.
Angle labels mark the boundaries between categories and the radial axis is measured
along the downward direction. Click legend entries to hide/show categories.


In [ ]:
# Bubble plot: marker size = % of the image occupied by the shade
interactive_polar(STATE['regions'], STATE['rows'], bubble=True).show()
# Scatter plot: fixed-size dots, easier to read when shades cluster densely
interactive_polar(STATE['regions'], STATE['rows'], bubble=False).show()

### 2.3 Individual color maps per category (bubble charts + segmentation)

In [ ]:
fig = shade_charts(STATE['im'], STATE['regions'])
plt.show()

### 2.4 Shade table (all detected colors: color, coordinates, name)

In [ ]:
df = pd.DataFrame(shades_to_records(STATE['rows']))
df.style.map(lambda h: f'background-color: {h}', subset=['hex'])

## 3 · Post-processing
### 3.1 Create clusters
Merge several categories into a single segmentation (e.g. all the blues).

In [ ]:
names = [r.name for r in STATE['regions']]
cluster_sel = W.SelectMultiple(options=names, value=('Blue', 'Ultramarine', 'Teal'),
                               rows=len(names), description='Categories')
cluster_out = W.Output()
cluster_btn = W.Button(description='Create cluster', button_style='info')

def do_cluster(_):
    with cluster_out:
        clear_output()
        mask, seg = make_cluster(STATE['im'], STATE['regions'], list(cluster_sel.value))
        STATE['last_cluster'] = (mask, seg)
        STATE['log'].append(f"make_cluster({list(cluster_sel.value)})")
        plt.figure(figsize=(6, 4)); plt.imshow(seg); plt.axis('off')
        plt.title(' + '.join(cluster_sel.value)); plt.show()

cluster_btn.on_click(do_cluster)
display(W.HBox([cluster_sel, W.VBox([cluster_btn, cluster_out])]))

### 3.2 Change boundaries
Define a custom region of the AB plane with sliders (θ₁, θ₂ in degrees, minimum radius).

In [ ]:
t1 = W.FloatSlider(value=90, min=0, max=360, description='θ1 (º)')
t2 = W.FloatSlider(value=180, min=0, max=360, description='θ2 (º)')
rmin = W.FloatSlider(value=30, min=0, max=130, description='ρ min')
bound_out = W.Output()

def do_bounds(change=None):
    with bound_out:
        clear_output(wait=True)
        mask, seg = custom_region(STATE['im'], t1.value, t2.value, rho_min=rmin.value)
        STATE['last_custom'] = (mask, seg)
        plt.figure(figsize=(6, 4)); plt.imshow(seg); plt.axis('off')
        plt.title(f'theta in ({t1.value:.0f},{t2.value:.0f}) deg, rho>{rmin.value:.0f} — {100*mask.mean():.1f}% of image')
        plt.show()

for w in (t1, t2, rmin):
    w.observe(do_bounds, names='value')
display(W.VBox([t1, t2, rmin, bound_out])); do_bounds()

### 3.3 Automatic clustering of shades on the AB plane
Cluster the detected shades directly in AB coordinates (the polar figure's plane).
Pick a method and, where applicable, the number of clusters. *Affinity Propagation,
DBSCAN, Mean Shift and OPTICS estimate the number of clusters themselves — the
slider is ignored for those (DBSCAN/OPTICS may also flag* noise *points, shown as
grey open circles). The plot keeps the category fan for context and colors each
shade by its cluster; the table summarizes each cluster's size, share of the image
and dominant categories.


In [ ]:
from abanicco import cluster_shades, cluster_summary, cluster_polar, CLUSTER_METHODS, AUTO_K_METHODS

cl_method = W.Dropdown(options=CLUSTER_METHODS, value='K-Means', description='Method')
cl_k      = W.IntSlider(value=4, min=2, max=15, description='# clusters')
cl_btn    = W.Button(description='Cluster', button_style='primary')
cl_out    = W.Output()

def _toggle_k(change=None):
    cl_k.disabled = cl_method.value in AUTO_K_METHODS
cl_method.observe(_toggle_k, names='value')
_toggle_k()

def run_clustering(_):
    with cl_out:
        clear_output()
        rows = STATE['rows']
        labels = cluster_shades(rows, cl_method.value, n_clusters=cl_k.value)
        STATE['cluster_labels'] = labels
        STATE['log'] = STATE.get('log', []) + [
            f'cluster_shades(method={cl_method.value!r}, n_clusters={cl_k.value})']
        cluster_polar(STATE['regions'], rows, labels,
                      title=f'{cl_method.value} on the AB plane').show()
        display(pd.DataFrame(cluster_summary(rows, labels)))

cl_btn.on_click(run_clustering)
display(W.VBox([W.HBox([cl_method, cl_k, cl_btn]), cl_out]))

# Auto-run once with the defaults so 'Run All' shows a result.
if 'cluster_labels' not in STATE:
    run_clustering(None)


### 3.4 Change number of colors per category / morphology
Just move the sliders in section 1 and press **Analyze** again — the whole
pipeline reruns and every plot refreshes when re-executed.

## 4 · Export
Tick what you want to save (mirrors the FIJI export checklist).

In [ ]:
chk_polar = W.Checkbox(value=True,  description='Polar description (PNG + HTML)')
chk_segs  = W.Checkbox(value=True,  description='Segmentations of each category (PNG)')
chk_ind   = W.Checkbox(value=False, description='Individual color masks of each category (PNG)')
chk_table = W.Checkbox(value=True,  description='Table of all detected colors (CSV)')
chk_log   = W.Checkbox(value=True,  description='List of modifications (TXT)')
chk_code  = W.Checkbox(value=True,  description='Python code to reproduce (PY)')
exp_dir   = W.Text(value='abanicco_output', description='Folder')
exp_btn   = W.Button(description='Export', button_style='success')
exp_out   = W.Output()

def do_export(_):
    with exp_out:
        clear_output()
        d = exp_dir.value; os.makedirs(d, exist_ok=True)
        im, regions, rows = STATE['im'], STATE['regions'], STATE['rows']
        if chk_polar.value:
            polar_description(regions, rows).savefig(f'{d}/polar_description.png', dpi=150)
            interactive_polar(regions, rows).write_html(f'{d}/polar_description.html')
            print('saved: polar description')
        if chk_segs.value:
            for r in regions:
                if r.segmented is not None:
                    Image.fromarray(r.segmented).save(f'{d}/segmentation_{r.name.replace(" ", "_")}.png')
            print('saved: category segmentations')
        if chk_ind.value:
            for r in regions:
                for k, s in enumerate(r.shades):
                    Image.fromarray((s['mask'] * 255).astype(np.uint8)).save(
                        f'{d}/{r.name.replace(" ", "_")}_shade{k+1}.png')
            print('saved: individual color masks')
        if chk_table.value:
            pd.DataFrame(shades_to_records(rows)).to_csv(f'{d}/shades.csv', index=False)
            print('saved: shade table')
        if chk_log.value:
            open(f'{d}/modifications.txt', 'w').write('\n'.join(STATE.get('log', [])))
            print('saved: modification log')
        if chk_code.value:
            code = (
                '# Auto-generated by the ABANICCO notebook - batch-ready\n'
                'import numpy as np\n'
                'from PIL import Image\n'
                'from abanicco import analyze, polar_description, shades_to_records\n'
                'import pandas as pd\n\n'
                f'im = np.array(Image.open({STATE["path"]!r}).convert("RGB"))\n'
                f'regions, rows = analyze(im, numshades={numshades.value}, morpho={morpho.value})\n'
                "pd.DataFrame(shades_to_records(rows)).to_csv('shades.csv', index=False)\n"
                "polar_description(regions, rows).savefig('polar.png', dpi=150)\n")
            open(f'{d}/reproduce.py', 'w').write(code)
            print('saved: reproducible code')
        print(f'All output in: {os.path.abspath(d)}')

exp_btn.on_click(do_export)
display(W.VBox([chk_polar, chk_segs, chk_ind, chk_table, chk_log, chk_code,
                W.HBox([exp_dir, exp_btn]), exp_out]))